# Visual Table Assistant — Training

## Purpose

This notebook is **only for training**. It does not prepare datasets from
scratch.

Flow:

1. Pull the packaged dataset ZIP `datasets/table_assistant_yolo_package.zip`
   from the DVC remote.
2. Extract the ZIP and restore the flat YOLO layout under
   `datasets/table_assistant_yolo/{images,labels}/` using symlink
3. Generate `reports/dataset_splits/{train,val,test}.txt` deterministically
   in this Colab runtime (rarest-class-per-image stratification, seed 42,
   ratios 60/15/25).
4. Validate dataset integrity and split files.
5. Configure MLflow.
6. Run a smoke training pass with YOLO.

## Prerequisites

Before running this notebook, make sure that:

1. **Dataset package is tracked with DVC**: `01_dataset_prep_colab.ipynb` was run, the resulting `datasets/table_assistant_yolo_package.zip` was added to DVC locally (`dvc add` + `dvc push`), and `datasets/table_assistant_yolo_package.zip.dvc` was committed and pushed to git.
2. **Colab Secrets are configured**. In the left sidebar of Colab open the key icon and add:

   - `GDRIVE_CREDENTIALS_DATA`

   This secret must contain the full JSON content of the cached DVC Google Drive credentials generated from a successful local authentication.
3. The Google account used to mount Drive in this session has access to the DVC remote folder.

## 1. Repository setup

Clone the repo (or pull the latest changes if it already exists from a previous
session) into `/content/iaa-visual-table-assistant`.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/LucasGVallejos/iaa-visual-table-assistant.git"
REPO_DIR = Path("/content/iaa-visual-table-assistant")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/iaa-table-assistant")
MLFLOW_DIR = DRIVE_PROJECT_DIR / "mlflow"
YOLO_OUTPUTS_DIR = DRIVE_PROJECT_DIR / "training_outputs"

In [ ]:
%cd /content

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already present, pulling latest changes...")
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}

## 2. Dependency installation

Install the project's pip requirements. `ultralytics`, `mlflow` and `dvc[gdrive]`
are all declared there.

In [ ]:
!pip install -q -r requirements.txt

## 3. Google Drive mount

Mount Drive so MLflow runs and DVC credentials/cache can persist across Colab
sessions. The DVC remote also resolves through this same mount when running
OAuth authentication.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
YOLO_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"MLFLOW_DIR:       {MLFLOW_DIR}")
print(f"YOLO_OUTPUTS_DIR: {YOLO_OUTPUTS_DIR}")

## 4. DVC pull from the Google Drive remote

This notebook pulls the packaged dataset ZIP from the DVC remote.

To avoid the interactive Google OAuth flow in Colab, DVC credentials are loaded from a Colab Secret named `GDRIVE_CREDENTIALS_DATA`. This secret must contain the JSON credentials previously generated by a successful local DVC authentication.

The notebook pulls only:

`datasets/table_assistant_yolo_package.zip.dvc`

and restores:

`datasets/table_assistant_yolo_package.zip`

In [ ]:
from google.colab import userdata
import os

gdrive_credentials = userdata.get("GDRIVE_CREDENTIALS_DATA")

if not gdrive_credentials:
    raise RuntimeError(
        "Missing Colab Secret: GDRIVE_CREDENTIALS_DATA. "
        "Create it from the cached DVC Google Drive credentials JSON."
    )

os.environ["GDRIVE_CREDENTIALS_DATA"] = gdrive_credentials

print("DVC Google Drive credentials loaded from Colab Secret.")

In [ ]:
PACKAGE_DVC = REPO_DIR / "datasets" / "table_assistant_yolo_package.zip.dvc"
PACKAGE_ZIP = REPO_DIR / "datasets" / "table_assistant_yolo_package.zip"

assert PACKAGE_DVC.exists(), f"Missing DVC file: {PACKAGE_DVC}"

!dvc pull datasets/table_assistant_yolo_package.zip.dvc

assert PACKAGE_ZIP.exists(), f"Missing DVC package zip: {PACKAGE_ZIP}"

size_gb = PACKAGE_ZIP.stat().st_size / (1024 ** 3)
print(f"Found package zip: {PACKAGE_ZIP} ({size_gb:.2f} GB)")

## 5. Extract packaged dataset

The DVC package ZIP is extracted and exposed through the standard dataset path by running the restore script.

After extraction, the package contains:

```text
datasets/table_assistant_yolo_package/
├── table_assistant_yolo/
│   ├── images/
│   └── labels/
└── metadata/
```


In [ ]:
!python -m src.data.preparation.restore_dataset_package

## 6. Generate deterministic split files

This step generates the YOLO split files from the restored flat-layout dataset.

The dataset itself is not physically split into `train/`, `val/` and `test/` folders. Instead, the script writes three text files:

- `reports/dataset_splits/train.txt`
- `reports/dataset_splits/val.txt`
- `reports/dataset_splits/test.txt`

Each file contains absolute image paths for the current Colab runtime. YOLO reads these paths through `configs/data_runtime_colab.yaml` and resolves labels by replacing `images/` with `labels/`.

The split strategy is deterministic:

- train: 60%
- validation: 15%
- test: 25%
- seed: 42
- stratification: rarest class present in each image

In [ ]:
!python -m src.data.preparation.split_dataset

## 7. MLflow setup

Configure MLflow with a tracking URI on Drive so all runs (params, metrics,
artifacts) survive Colab session resets. Ultralytics is told to log to MLflow
via its built-in integration.

In [ ]:
import os
import mlflow
from ultralytics import settings

MLFLOW_EXPERIMENT_NAME = "visual-table-assistant"
MLFLOW_TRACKING_URI = MLFLOW_DIR.as_uri()

MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

settings.update({"mlflow": True})

print("MLflow configured.")
print(f"Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment: {MLFLOW_EXPERIMENT_NAME}")

## 8. GPU check

Confirm a GPU is attached to this Colab runtime. YOLO will fall back to CPU
if none is available, but training would be unbearably slow, so the cell
warns loudly when CUDA is missing.

In [ ]:
!nvidia-smi

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "[WARN] CUDA is not available in this runtime. YOLO will run on CPU "
        "and training will be extremely slow. "
        "Switch to a GPU runtime: Runtime > Change runtime type > GPU."
    )

## 9. Smoke training

Run a short training pass on `yolov8n` to verify the entire pipeline (data_runtime_colab.yaml
→ split files → YOLO loader → training loop → MLflow logging) works end to end.

Five epochs is intentionally short. The goal is to confirm that nothing is
broken before kicking off the real baseline; metrics here are not meaningful.

In [ ]:
import os

from ultralytics import YOLO

os.environ["MLFLOW_RUN"] = "smoke_yolov8n_001"

smoke_model = YOLO("yolov8n.pt")

smoke_results = smoke_model.train(
    data=str(REPO_DIR / "configs" / "data_runtime_colab.yaml"),
    epochs=5,
    imgsz=640,
    batch=16,
    project=str(YOLO_OUTPUTS_DIR),
    name="smoke_yolov8n_001",
    exist_ok=True,
)

print("Smoke training finished.")
print(f"Run dir: {smoke_results.save_dir}")

## 10. Baseline training (placeholder)